# Task 020: lenstronomy_simple_ring — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Gravitational lensing mass reconstruction for simple Einstein ring using MCMC

**Paper**: lenstronomy II: A gravitational lensing software ecosystem
**Repository**: https://github.com/lenstronomy/lenstronomy

### Physical Background

Bayesian inversion provides posterior distributions, quantifying uncertainty alongside point estimates.

### Forward Model

```
y = G(m) + epsilon
```

### Inverse Problem

```
p(m|y) proportional to p(y|m) p(m)  via MCMC sampling
```

### Performance Metrics
- **PSNR**: 21.69 dB (model fit)
- **SSIM**: 0.3304
- **Evaluation**: Lens model parameter error (gravitational lensing)

### Mathematical Formulation

Gravitational lensing bends light according to the lens equation:

$$\boldsymbol{\beta} = \boldsymbol{\theta} - \boldsymbol{\alpha}(\boldsymbol{\theta})$$

where $\boldsymbol{\beta}$ is the source position, $\boldsymbol{\theta}$ is the image position, and $\boldsymbol{\alpha}$ is the deflection angle.

**Convergence**: $\kappa(\boldsymbol{\theta}) = \frac{\Sigma(\boldsymbol{\theta})}{\Sigma_{\text{cr}}}$

**SIE lens model** deflection:
$$\alpha_1 = \frac{\theta_E}{\sqrt{1-q^2}} \arctan\left(\frac{\sqrt{1-q^2}\,\theta_1}{\psi + q^2 s}\right)$$

**Source reconstruction**: $\hat{s} = \arg\min_s \|I_{\text{obs}} - L(\boldsymbol{\theta}_{\text{lens}}) s\|^2 + \lambda \|\nabla s\|_1$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import time
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Workflow.fitting_sequence import FittingSequence

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
def load_and_preprocess_data(
    numPix: int,
    pixel_scale: float,
    background_rms: float,
    exp_time: float,
    fwhm: float,
    psf_type: str,
    lens_model_list: list,
    kwargs_lens: list,
    source_model_list: list,
    kwargs_source: list,
    lens_light_model_list: list,
    kwargs_lens_light: list,
    random_seed: int = None
) -> dict:
    """
    Load and preprocess data: create simulated lensed image with noise.
    
    Returns a dictionary containing all data and model objects needed for fitting.
    """
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Define coordinate system
    transform_pix2angle = np.array([[-pixel_scale, 0], [0, pixel_scale]])
    
    # Calculate the RA/Dec of the pixel (0,0) such that the image is centered at (0,0)
    cx = (numPix - 1) / 2.0
    cy = (numPix - 1) / 2.0
    ra_at_xy_0 = -(transform_pix2angle[0, 0] * cx + transform_pix2angle[0, 1] * cy)
    dec_at_xy_0 = -(transform_pix2angle[1, 0] * cx + transform_pix2angle[1, 1] * cy)
    
    # Create kwargs_data
    kwargs_data = {
        'background_rms': background_rms,
        'exposure_time': exp_time,
        'ra_at_xy_0': ra_at_xy_0,
        'dec_at_xy_0': dec_at_xy_0,
        'transform_pix2angle': transform_pix2angle,
        'image_data': np.zeros((numPix, numPix))
    }
    
    # Create data class
    data_class = ImageData(**kwargs_data)
    
    # Create PSF
    kwargs_psf = {
        'psf_type': psf_type,
        'fwhm': fwhm,
        'pixel_size': pixel_scale,
        'truncation': 3
    }
    psf_class = PSF(**kwargs_psf)
    
    # Numerics settings
    kwargs_numerics = {
        'supersampling_factor': 1,
        'supersampling_convolution': False
    }
    
    # Create model classes
    lens_model_class = LensModel(lens_model_list)
    source_model_class = LightModel(source_model_list)
    lens_light_model_class = LightModel(lens_light_model_list)
    
    # Create image model for simulation
    imageModel = ImageModel(
        data_class, psf_class,
        lens_model_class=lens_model_class,
        source_model_class=source_model_class,
        lens_light_model_class=lens_light_model_class,
        kwargs_numerics=kwargs_numerics
    )
    
    # Generate noise-free model image
    image_model = imageModel.image(
        kwargs_lens, kwargs_source,
        kwargs_lens_light=kwargs_lens_light,
        kwargs_ps=None
    )
    
    # Add Poisson Noise (Photon Shot Noise)
    image_model_counts = image_model * exp_time
    image_model_counts[image_model_counts < 0] = 0
    poisson_counts = np.random.poisson(image_model_counts)
    image_with_poisson = poisson_counts / exp_time
    
    # Add Gaussian Background Noise
    bkg_noise = np.random.randn(*image_model.shape) * background_rms
    
    # Final simulated image
    image_real = image_with_poisson + bkg_noise
    
    # Update data class with simulated image
    data_class.update_data(image_real)
    kwargs_data['image_data'] = image_real
    
    print("Simulated Image Generated.")
    
    return {
        'image_data': image_real,
        'kwargs_data': kwargs_data,
        'kwargs_psf': kwargs_psf,
        'kwargs_numerics': kwargs_numerics,
        'lens_model_list': lens_model_list,
        'source_model_list': source_model_list,
        'lens_light_model_list': lens_light_model_list,
        'data_class': data_class,
        'psf_class': psf_class,
        'lens_model_class': lens_model_class,
        'source_model_class': source_model_class,
        'lens_light_model_class': lens_light_model_class,
        'true_kwargs_lens': kwargs_lens,
        'true_kwargs_source': kwargs_source,
        'true_kwargs_lens_light': kwargs_lens_light,
        'numPix': numPix,
        'pixel_scale': pixel_scale,
        'background_rms': background_rms,
        'exp_time': exp_time
    }

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = G(m) + epsilon
```

Functions: `forward_operator`

In [ ]:
def forward_operator(
    kwargs_lens: list,
    kwargs_source: list,
    kwargs_lens_light: list,
    data_class: ImageData,
    psf_class: PSF,
    lens_model_class: LensModel,
    source_model_class: LightModel,
    lens_light_model_class: LightModel,
    kwargs_numerics: dict
) -> np.ndarray:
    """
    Forward operator: compute model image given lens, source, and lens light parameters.
    
    This creates an ImageModel and computes the predicted image.
    """
    imageModel = ImageModel(
        data_class, psf_class,
        lens_model_class=lens_model_class,
        source_model_class=source_model_class,
        lens_light_model_class=lens_light_model_class,
        kwargs_numerics=kwargs_numerics
    )
    
    y_pred = imageModel.image(
        kwargs_lens, kwargs_source,
        kwargs_lens_light=kwargs_lens_light,
        kwargs_ps=None
    )
    
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
p(m|y) proportional to p(y|m) p(m)  via MCMC sampling
```

Functions: `run_inversion`

In [ ]:
def run_inversion(preprocessed_data: dict, fitting_kwargs_list: list) -> dict:
    """
    Run the inversion/fitting sequence using lenstronomy's FittingSequence.
    
    Returns the best fit parameters and chain list.
    """
    # Extract needed data
    kwargs_data = preprocessed_data['kwargs_data']
    kwargs_psf = preprocessed_data['kwargs_psf']
    kwargs_numerics = preprocessed_data['kwargs_numerics']
    lens_model_list = preprocessed_data['lens_model_list']
    source_model_list = preprocessed_data['source_model_list']
    lens_light_model_list = preprocessed_data['lens_light_model_list']
    
    # Setup lens parameters
    fixed_lens = []
    kwargs_lens_init = []
    kwargs_lens_sigma = []
    kwargs_lower_lens = []
    kwargs_upper_lens = []
    
    # SIE parameters
    fixed_lens.append({})
    kwargs_lens_init.append({
        'theta_E': 0.7, 'e1': 0., 'e2': 0.,
        'center_x': 0., 'center_y': 0.
    })
    kwargs_lens_sigma.append({
        'theta_E': 0.2, 'e1': 0.05, 'e2': 0.05,
        'center_x': 0.05, 'center_y': 0.05
    })
    kwargs_lower_lens.append({
        'theta_E': 0.01, 'e1': -0.5, 'e2': -0.5,
        'center_x': -10, 'center_y': -10
    })
    kwargs_upper_lens.append({
        'theta_E': 10., 'e1': 0.5, 'e2': 0.5,
        'center_x': 10, 'center_y': 10
    })
    
    # SHEAR parameters
    fixed_lens.append({'ra_0': 0, 'dec_0': 0})
    kwargs_lens_init.append({'gamma1': 0., 'gamma2': 0.0})
    kwargs_lens_sigma.append({'gamma1': 0.1, 'gamma2': 0.1})
    kwargs_lower_lens.append({'gamma1': -0.2, 'gamma2': -0.2})
    kwargs_upper_lens.append({'gamma1': 0.2, 'gamma2': 0.2})
    
    lens_params = [
        kwargs_lens_init, kwargs_lens_sigma, fixed_lens,
        kwargs_lower_lens, kwargs_upper_lens
    ]
    
    # Setup source parameters
    fixed_source = []
    kwargs_source_init = []
    kwargs_source_sigma = []
    kwargs_lower_source = []
    kwargs_upper_source = []
    
    fixed_source.append({})
    kwargs_source_init.append({
        'R_sersic': 0.2, 'n_sersic': 1, 'e1': 0, 'e2': 0,
        'center_x': 0., 'center_y': 0, 'amp': 16
    })
    kwargs_source_sigma.append({
        'n_sersic': 0.5, 'R_sersic': 0.1, 'e1': 0.05, 'e2': 0.05,
        'center_x': 0.2, 'center_y': 0.2, 'amp': 10
    })
    kwargs_lower_source.append({
        'e1': -0.5, 'e2': -0.5, 'R_sersic': 0.001, 'n_sersic': 0.5,
        'center_x': -10, 'center_y': -10, 'amp': 0
    })
    kwargs_upper_source.append({
        'e1': 0.5, 'e2': 0.5, 'R_sersic': 10, 'n_sersic': 5.,
        'center_x': 10, 'center_y': 10, 'amp': 100
    })
    
    source_params = [
        kwargs_source_init, kwargs_source_sigma, fixed_source,
        kwargs_lower_source, kwargs_upper_source
    ]
    
    # Setup lens light parameters
    fixed_lens_light = []
    kwargs_lens_light_init = []
    kwargs_lens_light_sigma = []
    kwargs_lower_lens_light = []
    kwargs_upper_lens_light = []
    
    fixed_lens_light.append({})
    kwargs_lens_light_init.append({
        'R_sersic': 0.5, 'n_sersic': 2, 'e1': 0, 'e2': 0,
        'center_x': 0., 'center_y': 0, 'amp': 16
    })
    kwargs_lens_light_sigma.append({
        'n_sersic': 1, 'R_sersic': 0.3, 'e1': 0.05, 'e2': 0.05,
        'center_x': 0.1, 'center_y': 0.1, 'amp': 10
    })
    kwargs_lower_lens_light.append({
        'e1': -0.5, 'e2': -0.5, 'R_sersic': 0.001, 'n_sersic': 0.5,
        'center_x': -10, 'center_y': -10, 'amp': 0
    })
    kwargs_upper_lens_light.append({
        'e1': 0.5, 'e2': 0.5, 'R_sersic': 10, 'n_sersic': 5.,
        'center_x': 10, 'center_y': 10, 'amp': 100
    })
    
    lens_light_params = [
        kwargs_lens_light_init, kwargs_lens_light_sigma, fixed_lens_light,
        kwargs_lower_lens_light, kwargs_upper_lens_light
    ]
    
    # Combine parameters
    kwargs_params = {
        'lens_model': lens_params,
        'source_model': source_params,
        'lens_light_model': lens_light_params
    }
    
    kwargs_likelihood = {'source_marg': False}
    kwargs_model = {
        'lens_model_list': lens_model_list,
        'source_light_model_list': source_model_list,
        'lens_light_model_list': lens_light_model_list
    }
    
    multi_band_list = [[kwargs_data, kwargs_psf, kwargs_numerics]]
    kwargs_data_joint = {
        'multi_band_list': multi_band_list,
        'multi_band_type': 'single-band'
    }
    kwargs_constraints = {'linear_solver': True}
    
    # Create fitting sequence
    fitting_seq = FittingSequence(
        kwargs_data_joint, kwargs_model, kwargs_constraints,
        kwargs_likelihood, kwargs_params, verbose=True
    )
    
    print("Starting Fitting Sequence...")
    start_time = time.time()
    chain_list = fitting_seq.fit_sequence(fitting_kwargs_list)
    kwargs_result = fitting_seq.best_fit()
    end_time = time.time()
    
    print(f"Fitting completed in {end_time - start_time:.2f} seconds.")
    
    return {
        'kwargs_result': kwargs_result,
        'chain_list': chain_list,
        'fitting_time': end_time - start_time,
        'fitting_seq': fitting_seq
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(
    inversion_result: dict,
    preprocessed_data: dict
) -> dict:
    """
    Evaluate the fitting results by comparing to true parameters and computing residuals.
    """
    kwargs_result = inversion_result['kwargs_result']
    
    print("Best Fit Result:")
    print(kwargs_result)
    
    # Extract fitted parameters
    fitted_kwargs_lens = kwargs_result.get('kwargs_lens', [])
    fitted_kwargs_source = kwargs_result.get('kwargs_source', [])
    fitted_kwargs_lens_light = kwargs_result.get('kwargs_lens_light', [])
    
    # Get true parameters
    true_kwargs_lens = preprocessed_data['true_kwargs_lens']
    true_kwargs_source = preprocessed_data['true_kwargs_source']
    true_kwargs_lens_light = preprocessed_data['true_kwargs_lens_light']
    
    # Compute model image with fitted parameters
    model_image = forward_operator(
        fitted_kwargs_lens,
        fitted_kwargs_source,
        fitted_kwargs_lens_light,
        preprocessed_data['data_class'],
        preprocessed_data['psf_class'],
        preprocessed_data['lens_model_class'],
        preprocessed_data['source_model_class'],
        preprocessed_data['lens_light_model_class'],
        preprocessed_data['kwargs_numerics']
    )
    
    # Compute residuals
    observed_image = preprocessed_data['image_data']
    residuals = observed_image - model_image
    
    # Compute chi-squared
    background_rms = preprocessed_data['background_rms']
    chi_squared = np.sum((residuals / background_rms) ** 2)
    reduced_chi_squared = chi_squared / (observed_image.size - 1)
    
    # Compare key lens parameters
    print("\n--- Parameter Comparison ---")
    if len(fitted_kwargs_lens) > 0 and len(true_kwargs_lens) > 0:
        print("Lens Model (SIE):")
        for key in ['theta_E', 'e1', 'e2', 'center_x', 'center_y']:
            if key in fitted_kwargs_lens[0] and key in true_kwargs_lens[0]:
                fitted_val = fitted_kwargs_lens[0][key]
                true_val = true_kwargs_lens[0][key]
                print(f"  {key}: True={true_val:.4f}, Fitted={fitted_val:.4f}, Diff={fitted_val - true_val:.4f}")
    
    if len(fitted_kwargs_lens) > 1 and len(true_kwargs_lens) > 1:
        print("Shear Model:")
        for key in ['gamma1', 'gamma2']:
            if key in fitted_kwargs_lens[1] and key in true_kwargs_lens[1]:
                fitted_val = fitted_kwargs_lens[1][key]
                true_val = true_kwargs_lens[1][key]
                print(f"  {key}: True={true_val:.4f}, Fitted={fitted_val:.4f}, Diff={fitted_val - true_val:.4f}")
    
    print(f"\nChi-squared: {chi_squared:.2f}")
    print(f"Reduced Chi-squared: {reduced_chi_squared:.4f}")
    print(f"Residual RMS: {np.std(residuals):.6f}")
    print(f"Fitting time: {inversion_result['fitting_time']:.2f} seconds")
    
    return {
        'model_image': model_image,
        'residuals': residuals,
        'chi_squared': chi_squared,
        'reduced_chi_squared': reduced_chi_squared,
        'residual_rms': np.std(residuals),
        'fitted_kwargs_lens': fitted_kwargs_lens,
        'fitted_kwargs_source': fitted_kwargs_source,
        'fitted_kwargs_lens_light': fitted_kwargs_lens_light
    }

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Configuration parameters
background_rms = 0.005
exp_time = 500.0
numPix = 60
pixel_scale = 0.05
fwhm = 0.05
psf_type = 'GAUSSIAN'

# Lens model configuration
lens_model_list = ['SIE', 'SHEAR']
kwargs_spemd = {
    'theta_E': 0.66, 'center_x': 0.05, 'center_y': 0,
    'e1': 0.07, 'e2': -0.03
}
kwargs_shear = {'gamma1': 0.0, 'gamma2': -0.05}
kwargs_lens = [kwargs_spemd, kwargs_shear]

# Source model configuration
source_model_list = ['SERSIC_ELLIPSE']
kwargs_sersic = {
    'amp': 16, 'R_sersic': 0.1, 'n_sersic': 1,
    'e1': -0.1, 'e2': 0.1, 'center_x': 0.1, 'center_y': 0
}
kwargs_source = [kwargs_sersic]

# Lens light model configuration
lens_light_model_list = ['SERSIC_ELLIPSE']
kwargs_sersic_lens = {
    'amp': 16, 'R_sersic': 0.6, 'n_sersic': 2,
    'e1': -0.1, 'e2': 0.1, 'center_x': 0.05, 'center_y': 0
}
kwargs_lens_light = [kwargs_sersic_lens]

# Fitting configuration (reduced iterations for demonstration)
fitting_kwargs_list = [
    ['PSO', {'sigma_scale': 1., 'n_particles': 50, 'n_iterations': 10}],
    ['MCMC', {'n_burn': 10, 'n_run': 10, 'n_walkers': 50, 'sigma_scale': 0.1}]
]

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
preprocessed_data = load_and_preprocess_data(
    numPix=numPix,
    pixel_scale=pixel_scale,
    background_rms=background_rms,
    exp_time=exp_time,
    fwhm=fwhm,
    psf_type=psf_type,
    lens_model_list=lens_model_list,
    kwargs_lens=kwargs_lens,
    source_model_list=source_model_list,
    kwargs_source=kwargs_source,
    lens_light_model_list=lens_light_model_list,
    kwargs_lens_light=kwargs_lens_light,
    random_seed=42
)

### Step 2: Run inversion/fitting

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 2: Run inversion/fitting
inversion_result = run_inversion(preprocessed_data, fitting_kwargs_list)

### Step 3: Evaluate results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 3: Evaluate results
evaluation = evaluate_results(inversion_result, preprocessed_data)

print("\nOPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **lenstronomy_simple_ring**:

1. **Problem Setup**: Bayesian inversion provides posterior distributions, quantifying uncertainty alongside point estimates.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.69 dB (model fit), SSIM=0.3304)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: lenstronomy II: A gravitational lensing software ecosystem
- Repository: https://github.com/lenstronomy/lenstronomy
